# Durable multi-agent orchestration on LangGraph + Agent Server

A walkthrough of a long-running **AI platform upgrade** workflow (phases P0–P6) run by a registry-driven orchestrator on Agent Server. It covers the scenarios you would also evaluate Temporal on:

| # | Scenario | What it shows |
|---|---|---|
| 1 | Registry → plan | The orchestrator is domain-agnostic. It plans a DAG with dependency gates from `registry/agents.yaml` |
| 2 | Run to the gates | P1 and P2 run in parallel, then **two gates pause in parallel** (`interrupt()`) |
| 3 | Approval | Resume **only** the approval gate with `Command(resume={interrupt_id: ...})`. The timer gate keeps waiting |
| 4 | Timed wake (10 days) | A scheduled `waker` (Agent Server cron or external trigger) resumes the timer. Includes a **live cron** check |
| 5 | Kill switch | A node dies mid-run. The thread resumes from the last checkpoint and the finished parallel step does **not** re-run. Also a real process kill (`os._exit`) |
| 6 | Lost response | A side effect commits but the response is lost. The retry is absorbed by **idempotency keys** |
| 7 | Saga compensation | The upgrade fails and compensators roll back to the known-good snapshot (*design pattern, not a built-in*) |
| 8 | Second use case | The same orchestrator runs TLS certificate rotation from the same registry |
| 9 | Observability | LangSmith traces and evaluators |

**Prereqs**: `uv sync`, then in another terminal `uv run langgraph dev` (local Agent Server on :2024). To use a deployment instead, set `LANGGRAPH_URL` and `LANGSMITH_API_KEY`.

In [1]:
import json, os, subprocess, sys, time, uuid
from pathlib import Path

from IPython.display import Markdown, display
from langgraph_sdk import get_sync_client

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
URL = os.environ.get("LANGGRAPH_URL", "http://127.0.0.1:2024")
WORKSPACE = os.environ.get("LANGSMITH_WORKSPACE_ID")  # needed with org-scoped service keys
client = get_sync_client(url=URL, api_key=os.environ.get("LANGSMITH_API_KEY"),
                         headers={"X-Tenant-Id": WORKSPACE} if WORKSPACE else None)
print("Agent Server:", URL, "| graphs:", sorted({a["graph_id"] for a in client.assistants.search()}))

Agent Server: http://127.0.0.1:2026 | graphs: ['orchestrator', 'planner', 'waker']


In [2]:
# ---- small helpers over the SDK ----------------------------------------------------------
RUN = dict(durability="sync")  # write each checkpoint before starting the next super-step

def new_cluster(prefix="aip-prod"):
    return f"{prefix}-{uuid.uuid4().hex[:4]}"

def upgrade_input(cluster_id, window="10d", faults=None):
    return {
        "goal": "Upgrade the AI platform cluster to 1.9.0",
        "params": {"cluster_id": cluster_id, "target_version": "1.9.0", "maintenance_window_delay": window},
        "faults": faults or {},
    }

def start(inputs):
    tid = client.threads.create()["thread_id"]
    client.runs.wait(tid, "orchestrator", input=inputs, raise_error=False, **RUN)
    return tid

def interrupts(tid):
    th = client.threads.get(tid)
    return [i for lst in (th.get("interrupts") or {}).values() for i in lst]

def show_interrupts(tid):
    th = client.threads.get(tid)
    print(f"thread status: {th['status']}")
    for i in interrupts(tid):
        v = i["value"]
        extra = f"wake_at={v['wake_at']}" if v["type"] == "timer" else f"role={v['approver_role']}"
        print(f"  waiting  {v['step_id']:24s} type={v['type']:9s} {extra}  interrupt_id={i['id'][:8]}…")

def wait_for_runs(tid):
    for r in client.runs.list(tid):
        if r["status"] in ("pending", "running"):
            client.runs.join(tid, r["run_id"])

def values(tid):
    return client.threads.get_state(tid)["values"]

def timeline(tid):
    for e in values(tid).get("events", []):
        extra = f"attempt={e['attempt']}" if e.get("attempt") else ""
        err = f"  error: {e['error']}" if e.get("error") else ""
        print(f"  round {e['round']:>2}  {e['step']:26s} {e['event']:24s} {extra}{err}")

def executions(tid):
    items = client.store.search_items(["demo_executions", tid], limit=100)["items"]
    return {i["key"]: i["value"]["count"] for i in sorted(items, key=lambda i: i["key"])}

def ledger(tid):
    items = client.store.search_items(["mock_platform", "ledger"], filter={"thread_id": tid}, limit=100)["items"]
    return {i["key"].split("/", 1)[1]: {k: i["value"][k] for k in ("requests", "applied")} for i in items}

def cluster(cid):
    v = client.store.get_item(["mock_platform", "clusters"], cid)["value"]
    return {k: v[k] for k in ("version", "control_plane_version", "health", "staged_bundle", "known_good")} | {
        "node_versions": [n["version"] for n in v["nodes"]]}

def approve(tid, approved=True, approver="cab-chair", comment="CHG-4711"):
    resume = {i["id"]: {"approved": approved, "approver": approver, "comment": comment}
              for i in interrupts(tid) if i["value"]["type"] == "approval"}
    client.runs.wait(tid, "orchestrator", command={"resume": resume}, raise_error=False, **RUN)

def run_waker(**kw):
    return client.runs.wait(None, "waker", input=kw)

def pp(x):
    print(json.dumps(x, indent=2, default=str))

## 1. Registry → plan (the orchestrator is domain-agnostic)

The orchestrator never names a platform phase. It reads `registry/agents.yaml`, where each agent declares the artifacts it `requires` and `provides`. The planner backward-chains from the goal artifact (`upgrade.verified`) and derives the **dependency gates** from those contracts. Steps in the same layer run in parallel.

In [3]:
sys.path.insert(0, str(ROOT / "src"))
from durable_orchestrator.registry import load_registry
from durable_orchestrator.planner import build_plan, plan_to_mermaid
from durable_orchestrator.timeutil import utcnow

reg = load_registry(ROOT / "registry" / "agents.yaml")
plan = build_plan(reg, goal="Upgrade the AI platform cluster to 1.9.0", target=None,
                  params={"cluster_id": "x", "target_version": "1.9.0", "maintenance_window_delay": "10d"}, now=utcnow())
print("goal artifact:", plan["target"], f"(chosen by {plan['planner']} planner)")
for n, layer in enumerate(plan["layers"]):
    print(f"  layer {n}: {layer}")
display(Markdown("```mermaid\n" + plan_to_mermaid(plan) + "\n```"))

goal artifact: upgrade.verified (chosen by keyword planner)
  layer 0: ['P0_discovery']
  layer 1: ['P1_precheck', 'P2_backup_verify']
  layer 2: ['P3_prepare_stage']
  layer 3: ['P4a_change_approval', 'P4b_maintenance_window']
  layer 4: ['P5_upgrade']
  layer 5: ['P6_verify_health']


```mermaid
flowchart LR
  P6_verify_health[P6_verify_health]
  P5_upgrade --> P6_verify_health
  P5_upgrade[P5_upgrade]
  P2_backup_verify --> P5_upgrade
  P3_prepare_stage --> P5_upgrade
  P4a_change_approval --> P5_upgrade
  P4b_maintenance_window --> P5_upgrade
  P2_backup_verify[P2_backup_verify]
  P0_discovery --> P2_backup_verify
  P0_discovery[P0_discovery]
  P3_prepare_stage[P3_prepare_stage]
  P1_precheck --> P3_prepare_stage
  P2_backup_verify --> P3_prepare_stage
  P1_precheck[P1_precheck]
  P0_discovery --> P1_precheck
  P4b_maintenance_window{{P4b_maintenance_window}}
  P3_prepare_stage --> P4b_maintenance_window
  P4a_change_approval{{P4a_change_approval}}
  P3_prepare_stage --> P4a_change_approval
```

## 2. Run to the gates: parallel phases, then two gates paused in parallel

One `runs.wait` call. The run goes through P0, runs **P1 pre-check and P2 backup in the same round** (parallel), stages the bundle in P3, and then pauses on **two parallel gates**:

* `P4a_change_approval`: a human approval (`interrupt()`)
* `P4b_maintenance_window`: a timer gate whose window opens **10 days** from now

While paused, the thread holds no worker. Its state is a checkpoint in Agent Server's database.

In [4]:
CID = new_cluster()
tid = start(upgrade_input(CID, window="10d"))
print("thread:", tid, "| cluster:", CID)
timeline(tid)
show_interrupts(tid)

thread: 01a21508-55ef-7645-ad45-15c21aa991b6 | cluster: aip-prod-d181
  round  0  plan                       planned                  
  round  1  P0_discovery               succeeded                attempt=1
  round  2  P2_backup_verify           succeeded                attempt=1
  round  2  P1_precheck                succeeded                attempt=1
  round  3  P3_prepare_stage           succeeded                attempt=1
thread status: interrupted
  waiting  P4b_maintenance_window   type=timer     wake_at=2026-12-05T23:26:38+00:00  interrupt_id=30ddeb0f…
  waiting  P4a_change_approval      type=approval  role=change-advisory-board  interrupt_id=dcfc07b8…


## 3. Approve one gate. The other keeps waiting.

Each paused gate has its own interrupt id. We resume **only** the approval with `Command(resume={approval_id: {...}})`. The timer gate re-suspends on its own, and nothing upstream re-runs: P0–P3 each ran exactly once.

The gate nodes show 2 executions. That is how `interrupt()` works: on resume, the node runs again from its start and `interrupt()` returns the resume value. Both gates in the super-step re-enter, and the one without a resume value suspends again. So code *before* an `interrupt()` must be side-effect free. The same rule applies to Temporal workflow code, which is replayed.

In [5]:
approve(tid)
show_interrupts(tid)
print("P4a result:", values(tid)["results"]["P4a_change_approval"]["artifacts"]["change.approved"])
print("executions:", executions(tid))

thread status: interrupted
  waiting  P4b_maintenance_window   type=timer     wake_at=2026-12-05T23:26:38+00:00  interrupt_id=30ddeb0f…
P4a result: {'approved': True, 'approver': 'cab-chair', 'comment': 'CHG-4711', 'decided_at': '2026-11-25T23:26:42+00:00'}
executions: {'P0_discovery': 1, 'P1_precheck': 1, 'P2_backup_verify': 1, 'P3_prepare_stage': 1, 'P4a_change_approval': 2, 'P4b_maintenance_window': 2}


## 4. Timed wake: a 10-day wait

`interrupt()` never wakes up on its own. It resumes only when something calls the thread. So the wake-up comes from outside: a small stateless **`waker` graph** that finds timer interrupts whose `wake_at` has passed and resumes exactly those interrupt ids. It runs on an **Agent Server cron**, or from any external trigger (`scripts/trigger_waker.py`).

First, run the waker with the real clock. The window is 10 days away, so nothing wakes:

In [6]:
pp(run_waker(thread_ids=[tid]))

{
  "thread_ids": [
    "01a21508-55ef-7645-ad45-15c21aa991b6"
  ],
  "now": "2026-11-25T23:26:43+00:00",
  "checked": 1,
  "woken": [],
  "waiting": [
    {
      "thread_id": "01a21508-55ef-7645-ad45-15c21aa991b6",
      "interrupt_id": "30ddeb0f9dcd21c326355853c8e7b15a",
      "step_id": "P4b_maintenance_window",
      "wake_at": "2026-12-05T23:26:38+00:00"
    }
  ]
}


We can't wait 10 days in a demo. The waker accepts a demo-only clock skew (`advance="10d"`). Nothing else changes: the same interrupt is resumed on the same thread, and the run picks up at P5.

In [7]:
pp(run_waker(thread_ids=[tid], advance="10d"))
wait_for_runs(tid)
timeline(tid)
print("\nfinal status:", values(tid)["status"])
pp(cluster(CID))

{
  "advance": "10d",
  "thread_ids": [
    "01a21508-55ef-7645-ad45-15c21aa991b6"
  ],
  "now": "2026-12-05T23:26:44+00:00",
  "checked": 1,
  "woken": [
    {
      "thread_id": "01a21508-55ef-7645-ad45-15c21aa991b6",
      "interrupt_ids": [
        "30ddeb0f9dcd21c326355853c8e7b15a"
      ]
    }
  ],
  "waiting": []
}


  round  0  plan                       planned                  
  round  1  P0_discovery               succeeded                attempt=1
  round  2  P2_backup_verify           succeeded                attempt=1
  round  2  P1_precheck                succeeded                attempt=1
  round  3  P3_prepare_stage           succeeded                attempt=1
  round  4  P4b_maintenance_window     succeeded                attempt=3
  round  4  P4a_change_approval        succeeded                attempt=2
  round  5  P5_upgrade                 succeeded                attempt=1
  round  6  P6_verify_health           succeeded                attempt=1
  round  7  finalize                   succeeded                

final status: succeeded
{
  "version": "1.9.0",
  "control_plane_version": "1.9.0",
  "health": "green",
  "staged_bundle": "1.9.0",
  "known_good": {
    "snapshot_id": "snap-pre-upgrade-ba99be94",
    "version": "1.8.2"
  },
  "node_versions": [
    "1.9.0",
    "1.9.0",
   

### 4b. Live check: a real Agent Server cron wakes a real wait

This is the end-to-end check with no clock tricks. The window opens **60 seconds** after planning, a cron runs the waker every minute, and we poll until the thread finishes by itself. It takes about 1–2 minutes.

In [8]:
CID_CRON = new_cluster("aip-cron")
tid_cron = start(upgrade_input(CID_CRON, window="60s"))
approve(tid_cron)
show_interrupts(tid_cron)

cron = client.crons.create("waker", schedule="* * * * *", input={})
print("cron registered:", cron["cron_id"], "next run:", cron.get("next_run_date"))
t0 = time.time()
try:
    while client.threads.get(tid_cron)["status"] != "idle" and time.time() - t0 < 240:
        time.sleep(5)
finally:
    client.crons.delete(cron["cron_id"])
print(f"thread finished after {time.time() - t0:.0f}s, status={values(tid_cron)['status']}")
print("P4b opened by:", values(tid_cron)["results"]["P4b_maintenance_window"]["artifacts"]["window.open"])

thread status: interrupted
  waiting  P4b_maintenance_window   type=timer     wake_at=2026-11-25T23:27:47+00:00  interrupt_id=ba127bd1…
cron registered: 01a21508-8c1e-7440-bd16-2dbc76ab40aa next run: 2026-11-25T23:27:00+00:00


thread finished after 75s, status=succeeded
P4b opened by: {'opened_at': '2026-11-25T23:28:03+00:00', 'opened_by': 'waker', 'override': False}


## 5. Kill switch: a node dies mid-run

The kill switch makes `P2_backup_verify` die **mid-node**. It has already taken a snapshot (a side effect) and is about to record the known-good state. At that moment its parallel sibling `P1_precheck` has already finished.

When a parallel task finishes, LangGraph saves its writes to the checkpoint right away. So after the crash the thread holds: P0 done, P1 done, P2 not done.

In [9]:
CID_KILL = new_cluster("aip-kill")
kill = {"kill": {"step": "P2_backup_verify", "before_tool": "platform_record_known_good", "mode": "raise"}}
tid_kill = start(upgrade_input(CID_KILL, faults=kill))
print("thread status after crash:", client.threads.get(tid_kill)["status"])
print("executions so far:      ", executions(tid_kill))
print("steps already committed:", sorted(values(tid_kill).get("results", {})))

thread status after crash: error
executions so far:       {'P0_discovery': 1, 'P1_precheck': 1, 'P2_backup_verify': 1}
steps already committed: ['P0_discovery', 'P1_precheck']


**Resume** = run the same thread with no input. Execution continues from the last checkpoint:

* `P1_precheck` stays at **1** execution. It finished before the crash and is not re-run.
* `P2_backup_verify` runs a 2nd time. Its snapshot call is replayed with the same idempotency key, so the downstream system returns the stored result and **no second snapshot** is taken (`requests=2, applied=1`).

In [10]:
client.runs.wait(tid_kill, "orchestrator", input=None, **RUN)
print("executions after resume:", executions(tid_kill))
print("take_snapshot ledger:   ", ledger(tid_kill)["P2_backup_verify/take_snapshot"])
print("pre-upgrade snapshots:  ",
      [s["id"] for s in client.store.get_item(["mock_platform", "clusters"], CID_KILL)["value"]["snapshots"] if s["kind"] == "pre-upgrade"])
show_interrupts(tid_kill)

executions after resume: {'P0_discovery': 1, 'P1_precheck': 1, 'P2_backup_verify': 2, 'P3_prepare_stage': 1, 'P4a_change_approval': 1, 'P4b_maintenance_window': 1}
take_snapshot ledger:    {'requests': 2, 'applied': 1}
pre-upgrade snapshots:   ['snap-pre-upgrade-55f896e8']
thread status: interrupted
  waiting  P4b_maintenance_window   type=timer     wake_at=2026-12-05T23:28:07+00:00  interrupt_id=0298bfee…
  waiting  P4a_change_approval      type=approval  role=change-advisory-board  interrupt_id=eb545d13…


### 5b. Hard kill: the process really dies

The server-safe kill above raises an exception. This version calls `os._exit(137)` mid-node, so the **process is gone**. A new process then resumes the thread from SQLite checkpoints. On Agent Server the same checkpoints live in Postgres.

In [11]:
def sh(*args):
    p = subprocess.run([sys.executable, "scripts/hard_kill_demo.py", *args], cwd=ROOT, capture_output=True, text=True)
    print(f"$ hard_kill_demo.py {' '.join(args)}   -> exit code {p.returncode}")
    print(p.stdout.strip() or p.stderr.strip()[-2000:])

sh("run")      # dies with exit code 137 inside P2
sh("resume")   # brand-new process continues; P1 still shows 1 execution

$ hard_kill_demo.py run   -> exit code 137
starting run; the kill switch will os._exit(137) inside P2_backup_verify ...


$ hard_kill_demo.py resume   -> exit code 0
new process: resuming thread from its last checkpoint ...
{
  "next": [
    "run_step",
    "run_step"
  ],
  "completed_steps": [
    "P0_discovery",
    "P1_precheck",
    "P2_backup_verify",
    "P3_prepare_stage"
  ],
  "waiting_on": [
    "P4b_maintenance_window",
    "P4a_change_approval"
  ],
  "executions": {
    "P4b_maintenance_window": 1,
    "P4a_change_approval": 1,
    "P3_prepare_stage": 1,
    "P2_backup_verify": 2,
    "P0_discovery": 1,
    "P1_precheck": 1
  }
}


## 6. Lost response, retried idempotently

`platform_upgrade_control_plane` applies the upgrade, then the connection drops before the response arrives (`TransientInfraError`). The node's `RetryPolicy` re-runs the step. Every side-effecting tool sends the same deterministic idempotency key (`thread/step/operation`), so the retry recovers the stored response and nothing is applied twice.

To get past the gates we resume **both** parallel gates in one command (approval + window override): `Command(resume={approval_id: ..., timer_id: ...})`.

In [12]:
def approve_and_open_window(tid):
    resume = {}
    for i in interrupts(tid):
        if i["value"]["type"] == "approval":
            resume[i["id"]] = {"approved": True, "approver": "cab-chair"}
        else:
            resume[i["id"]] = {"action": "override", "by": "ops-lead"}
    client.runs.wait(tid, "orchestrator", command={"resume": resume}, raise_error=False, **RUN)

CID_LOST = new_cluster("aip-lost")
tid_lost = start(upgrade_input(CID_LOST, faults={"lose_response": {"tool": "platform_upgrade_control_plane", "times": 1}}))
approve_and_open_window(tid_lost)
print("status:", values(tid_lost)["status"])
print("P5_upgrade executions:", executions(tid_lost)["P5_upgrade"])
pp({k: v for k, v in ledger(tid_lost).items() if k.startswith("P5_upgrade")})
print("cluster:", cluster(CID_LOST)["version"])

status: succeeded
P5_upgrade executions: 2
{
  "P5_upgrade/drain_nodes": {
    "requests": 2,
    "applied": 1
  },
  "P5_upgrade/upgrade_control_plane": {
    "requests": 2,
    "applied": 1
  },
  "P5_upgrade/upgrade_workers": {
    "requests": 1,
    "applied": 1
  },
  "P5_upgrade/uncordon_nodes": {
    "requests": 1,
    "applied": 1
  }
}
cluster: 1.9.0


## 7. Saga compensation after a failed upgrade

> **Design pattern, not a built-in feature.** LangGraph and Agent Server have no saga primitive. The policy lives in `src/durable_orchestrator/saga.py` (about 40 lines). The platform contributes the parts that are hard to get right: which steps committed (checkpoints), compensations as durable steps of their own, and safe re-runs (idempotent tools).

`platform_upgrade_workers` upgrades one worker, then fails non-retryably (`StepFailure`). The orchestrator stops forward work and runs compensators in reverse order: `C5_rollback` restores the known-good snapshot recorded in P2, then `C3_unstage` removes the staged bundle.

In [13]:
CID_SAGA = new_cluster("aip-saga")
tid_saga = start(upgrade_input(CID_SAGA, faults={"fail": {"tool": "platform_upgrade_workers"}}))
approve_and_open_window(tid_saga)
timeline(tid_saga)
r = values(tid_saga)["report"]
print("\nstatus:", r["status"], "| failed:", r["failed_steps"])
print("cluster after rollback:"); pp(cluster(CID_SAGA))

  round  0  plan                       planned                  
  round  1  P0_discovery               succeeded                attempt=1
  round  2  P2_backup_verify           succeeded                attempt=1
  round  2  P1_precheck                succeeded                attempt=1
  round  3  P3_prepare_stage           succeeded                attempt=1
  round  4  P4b_maintenance_window     succeeded                attempt=2
  round  4  P4a_change_approval        succeeded                attempt=2
  round  5  P5_upgrade                 failed                   attempt=1  error: aip-saga-b148-w2 failed to rejoin after upgrade
  round  6  undo:P5_upgrade            compensation_succeeded   attempt=1
  round  7  undo:P3_prepare_stage      compensation_succeeded   attempt=1
  round  8  finalize                   rolled_back              

status: rolled_back | failed: {'P5_upgrade': 'aip-saga-b148-w2 failed to rejoin after upgrade'}
cluster after rollback:
{
  "version": "1.8.2",
  "

## 8. Same orchestrator, different use case

TLS certificate rotation is a separate set of agents in the same registry. No orchestrator code changes. Here the goal is free text, and the planner maps it to the `certs.verified` goal. With `PLANNER_MODEL` set, an LLM does this mapping. Without it, keyword matching does.

In [14]:
CID_CERT = new_cluster("aip-certs")
tid_cert = start({"goal": "Rotate the TLS certificates on the cluster", "params": {"cluster_id": CID_CERT}})
print("plan layers:", values(tid_cert)["plan"]["layers"])
show_interrupts(tid_cert)
approve(tid_cert, approver="security-oncall")
print("status:", values(tid_cert)["status"])
print("certs:", client.store.get_item(["mock_platform", "clusters"], CID_CERT)["value"]["certs"])

plan layers: [['CR1_cert_inventory'], ['CR2_rotation_approval'], ['CR3_rotate'], ['CR4_verify_tls']]
thread status: interrupted
  waiting  CR2_rotation_approval    type=approval  role=security  interrupt_id=38a08221…


status: succeeded
certs: {'current': 'certs-fddab851', 'previous': 'certs-2025-q1', 'pending': None, 'expires_in_days': 397}


## 9. Observability: LangSmith traces + evaluators

With `LANGSMITH_API_KEY` / `LANGSMITH_TRACING=true` in `.env`, Agent Server traces every run: orchestrator super-steps, each sub-agent (`agent:P5_upgrade`, …), every tool call, and every retry. Resumes and waker runs appear as separate runs grouped by **thread** in LangSmith, so a 10-day workflow reads as one conversation.

The eval suite replays the failure scenarios in-process and scores properties a durable workflow must hold:

* `outcome_matches_expected`
* `dependency_gates_respected`: no step ran before its dependencies succeeded
* `side_effects_exactly_once`: every side effect applied once, despite retries and resumes
* `reached_known_good_state`: the target version, or rolled back to the snapshot

`--local` prints scores. Without the flag, the suite uploads a dataset and an experiment to LangSmith.

In [15]:
p = subprocess.run([sys.executable, "evals/run_evals.py", "--local"], cwd=ROOT, capture_output=True, text=True)
print(p.stdout[-4000:] or p.stderr[-4000:])


happy_path: status=succeeded crashes_recovered=0
  PASS  outcome_matches_expected     got succeeded
  PASS  dependency_gates_respected   8 steps in dependency order
  PASS  side_effects_exactly_once    7 side effects, 0 idempotent replays absorbed, duplicates=[]
  PASS  reached_known_good_state     cluster matches {'version': '1.9.0', 'control_plane_version': '1.9.0', 'health': 'green'}

kill_mid_node_then_resume: status=succeeded crashes_recovered=1
  PASS  outcome_matches_expected     got succeeded
  PASS  dependency_gates_respected   8 steps in dependency order
  PASS  side_effects_exactly_once    7 side effects, 1 idempotent replays absorbed, duplicates=[]
  PASS  reached_known_good_state     cluster matches {'version': '1.9.0', 'control_plane_version': '1.9.0', 'health': 'green'}

lost_response_retry: status=succeeded crashes_recovered=0
  PASS  outcome_matches_expected     got succeeded
  PASS  dependency_gates_respected   8 steps in dependency order
  PASS  side_effects_exactly

## 10. On top: a Managed Deep Agent operator

[`upgrade-operator/`](../upgrade-operator/) is a **Managed Deep Agent** that operates this orchestrator over the same SDK. It previews plans, starts workflows after a human OK (`interrupt_on`), answers CAB gates only with a named approver and a `CHG-` ticket (middleware policy), schedules its own follow-ups (`runtime.schedules`), investigates rollbacks with an async subagent, and keeps lessons in durable memory. See the "Managed Deep Agents layer" section of the README.

Set `OPERATOR_URL` to the operator deployment to run this cell. It asks a read-only question, so no HITL approval is needed. Try the full upgrade conversation in Studio.

In [16]:
OP_URL = os.environ.get("OPERATOR_URL")
if not OP_URL:
    print("OPERATOR_URL not set; skipping the operator demo.")
else:
    op = get_sync_client(url=OP_URL, api_key=os.environ.get("LANGSMITH_API_KEY"),
                         headers={"X-Tenant-Id": WORKSPACE} if WORKSPACE else None)
    op_thread = op.threads.create()["thread_id"]
    op.runs.wait(op_thread, "upgrade-operator",
                 input={"messages": [{"role": "user", "content": "Which AI platform workflows are waiting on something, and on whom?"}]})
    msgs = op.threads.get(op_thread)["values"]["messages"]
    print("tool calls:", [tc["name"] for m in msgs if m["type"] == "ai" for tc in (m.get("tool_calls") or [])])
    last = msgs[-1]["content"]
    print(last if isinstance(last, str) else "".join(b.get("text", "") for b in last if isinstance(b, dict)))

OPERATOR_URL not set; skipping the operator demo.


## What Agent Server handled vs. what we wrote

| Concern | Agent Server / LangGraph (platform) | This repo (application code) |
|---|---|---|
| Durable state per step, resume after crash | Checkpointer (Postgres), pending writes for finished parallel tasks | – |
| Human waits | `interrupt()` + `Command(resume=…)`, thread status, per-interrupt ids | Gate payload / validation (`gates.py`) |
| Long timers | Cron scheduler, stateless runs, in-process SDK client | `waker` graph (~60 lines) |
| Retries | `RetryPolicy` on the node | Error taxonomy (`errors.py`) |
| Idempotency | – | Deterministic keys + idempotent downstream API |
| Saga compensation | Durable steps to build it on | Policy (`saga.py`), *a design pattern* |
| Task queue, workers, HTTP API, streaming, auth | Built in | – |
| Tracing, datasets, experiments | LangSmith | Evaluators (`evals/`) |